In [4]:
# setup
import re, csv, html, time, requests
from pathlib import Path
from urllib.parse import urljoin, quote

out = Path("data/labins-index")
out.mkdir(parents=True, exist_ok=True)
logf = out / "log.csv"
page = "https://www.labins.org/survey_data/landrecords/glo_volsearch2.cfm?mvol={n:03d}"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}

# load checkpoint
done = set()
if logf.exists():
    with open(logf) as fh:
        done = {int(r["vol"]) for r in csv.DictReader(fh) if r["pdf_status"] == "200"}
else:
    logf.write_text("vol,page_status,pdf_url,pdf_status,bytes\n")
print(len(done), "volumes already done")

# get with retry on 5xx
def get(url):
    for i in range(3):
        r = requests.get(url, headers=hdr, timeout=60)
        if r.status_code < 500:
            return r
        time.sleep(2 * (i + 1))
    return r

# fetch pages then pdfs
for n in range(1, 279):
    if n in done:
        continue
    p = get(page.format(n=n))
    print(n, "page", p.status_code)
    url, pst, size = "", "", 0
    if p.status_code == 200:
        # find link by anchor text
        m = re.search(r'<a[^>]*href=["\']([^"\']+)["\'][^>]*>\s*All Index and Diagram Pages', p.text, re.I | re.S)
        if m:
            url = quote(urljoin(p.url, html.unescape(m.group(1))), safe=":/&%?=")   # encode spaces
            r = get(url)
            pst = str(r.status_code)
            print(n, "pdf", r.status_code, url)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                (out / f"vol{n:03d}.pdf").write_bytes(r.content)
                size = len(r.content)
            elif r.status_code == 200:
                pst = "not pdf"
        else:
            pst = "no link"
            print(n, "no pdf link")
    with open(logf, "a", newline="") as fh:   # checkpoint per volume
        csv.writer(fh).writerow([n, p.status_code, url, pst, size])
    if p.status_code >= 400 or (pst.isdigit() and int(pst) >= 400):   # stop on any 4xx
        print("stopping at", n)
        break
    time.sleep(2)   # rate limit

# summary
print(len(list(out.glob("vol*.pdf"))), "files saved")

274 volumes already done
83 page 200
83 no pdf link
103 page 200
103 no pdf link
192 page 200
192 no pdf link
217 page 200
217 pdf 404 https://ftp.labins.org/glo_all/Volume217_pdf/Folder%20%201%20Inex%20&%20Diagram_pdf.pdf
stopping at 217
274 files saved


In [5]:
# setup
import csv, time, requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

out = Path("data/labins-index")
logf = out / "log.csv"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}

# manual urls
manual = {
    83: "https://ftp.labins.org/glo_all/Volume83_pdf/Folder%20%201%20Index%20&%20Diagarm_pdf.pdf",
    103: "https://ftp.labins.org/glo_all/Volume103_pdf/Folder%20%201%20Index%20&%20Diagame_pdf.pdf"
}

# load log
log = pd.read_csv(logf, dtype={"pdf_url": str, "pdf_status": str}).fillna("")

# fetch manual urls and update log
for n, u in manual.items():
    u = quote(u, safe=":/&%?=")   # encode spaces
    r = requests.get(u, headers=hdr, timeout=60)
    ok = r.status_code == 200 and r.content[:4] == b"%PDF"
    print(n, r.status_code, "pdf" if ok else "not pdf", u)
    if ok:
        (out / f"vol{n:03d}.pdf").write_bytes(r.content)
    row = {"vol": n, "page_status": 200, "pdf_url": u,
           "pdf_status": "200" if ok else str(r.status_code), "bytes": len(r.content) if ok else 0}
    log = log[log["vol"] != n]   # drop old row for this vol
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    time.sleep(1)   # rate limit
log = log.sort_values("vol").reset_index(drop=True)
log.to_csv(logf, index=False)

# remaining missingness
bad = log[(log["pdf_status"] != "200") | (~log["vol"].map(lambda n: (out / f"vol{n:03d}.pdf").exists()))]
gaps = sorted(set(range(1, int(log["vol"].max()) + 1)) - set(log["vol"]))   # vols never logged
print(len(log), "rows,", len(bad), "missing pdf")
print(bad[["vol", "page_status", "pdf_status", "pdf_url"]].to_string(index=False))
print("not in log:", gaps)

83 200 pdf https://ftp.labins.org/glo_all/Volume83_pdf/Folder%20%201%20Index%20&%20Diagarm_pdf.pdf
103 200 pdf https://ftp.labins.org/glo_all/Volume103_pdf/Folder%20%201%20Index%20&%20Diagame_pdf.pdf
278 rows, 2 missing pdf
 vol  page_status pdf_status                                                                                 pdf_url
 192          200    no link                                                                                        
 217          200        404 https://ftp.labins.org/glo_all/Volume217_pdf/Folder%20%201%20Inex%20&%20Diagram_pdf.pdf
not in log: []


## Crawling Index Pages (/glo_all/ and /DRG2/)

In [3]:
# setup
import csv, html, re, time, requests
import pandas as pd
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import urljoin, unquote, urlparse

out = Path("data/labins/site/")
root = "https://ftp.labins.org/glo_all/"
name = "glo_all"
pathsf, logf, treef = out / f"{name}_paths.csv", out / f"{name}_log.csv", out / f"{name}_tree.txt"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}
delay = 1.0  # seconds between requests
cols = ["path", "url", "type", "size", "bytes", "modified", "depth"]
base_depth = urlparse(root).path.rstrip("/").count("/")

# patterns
link = re.compile(r'<a href="([^"]*)">(.*?)</a>(.*?)(?=<a href=|</pre>|</table>|\Z)', re.S | re.I)
stamp = re.compile(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}")
mult = {"K": 1024, "M": 1024**2, "G": 1024**3, "T": 1024**4}

# listing sizes like 55K to approx bytes
def tobytes(s):
    return float(s[:-1]) * mult[s[-1]] if s[-1:] in mult else float(s) if re.fullmatch(r"\d+(\.\d+)?", s) else 0

# parse one listing page
def parse(text, base):
    rows = []
    for href, label, tail in link.findall(text):
        if href.startswith("?") or "Parent Directory" in label:
            continue   # sort links and the way back up
        url = urljoin(base, html.unescape(href))
        if not url.startswith(base) or url == base:
            continue
        tail = html.unescape(re.sub(r"<[^>]+>", " ", tail)).replace("\xa0", " ")
        d = stamp.search(tail)
        rest = tail[d.end():].split() if d else []
        size = rest[0] if rest else ""
        path = unquote(urlparse(url).path)
        rows.append({"path": path, "url": url, "type": "dir" if url.endswith("/") else "file", "size": size,
                     "bytes": tobytes(size), "modified": d.group() if d else "", "depth": path.rstrip("/").count("/") - base_depth})
    return rows

# session reused across requests
def new_session():
    s = requests.Session()
    s.headers.update(hdr)
    return s
session = new_session()

# get with retry on 5xx and connection errors
def get(url):
    global session
    r = None
    for i in range(4):
        try:
            r = session.get(url, timeout=(15, 60))   # connect, read
            if r.status_code < 500:
                return r
            time.sleep(2 * (i + 1))
        except requests.exceptions.RequestException as e:
            print("connection problem:", type(e).__name__, "- retry", i + 1)
            session.close()
            session = new_session()   # drop the dead connection
            r = None
            time.sleep(10 * (i + 1))
    return r if r is not None else SimpleNamespace(status_code=0, text="")   # 0 means no connection

# load checkpoint
out.mkdir(parents=True, exist_ok=True)
if logf.exists() and pathsf.exists():
    log = pd.read_csv(logf, dtype=str).fillna("")
    done = set(log.loc[log["status"] == "200", "url"])
    known = pd.read_csv(pathsf, dtype=str).fillna("")
    seen = set(known.loc[known["type"] == "dir", "url"]) | {root}
else:
    pathsf.write_text(",".join(cols) + "\n")
    logf.write_text("url,status,dirs,files\n")
    done, seen = set(), {root}
stack = [u for u in sorted(seen, key=lambda u: (u != root, u)) if u not in done]
print(len(done), "directories already done,", len(stack), "to go")

# crawl
while stack:
    u = stack.pop(0)
    r = get(u)
    print(r.status_code, u)
    nd = nf = 0
    if r.status_code == 200:
        rows = parse(r.text, u)
        with open(pathsf, "a", newline="") as fh:   # checkpoint per directory
            csv.DictWriter(fh, cols).writerows(rows)
        for x in rows:
            nd, nf = nd + (x["type"] == "dir"), nf + (x["type"] == "file")
            if x["type"] == "dir" and x["url"] not in seen:
                seen.add(x["url"])
                stack.append(x["url"])
    with open(logf, "a", newline="") as fh:
        csv.writer(fh).writerow([u, r.status_code, nd, nf])
    if r.status_code == 0 or r.status_code >= 400:   # stop on no connection or any 4xx
        print("stopping at", u)
        break
    time.sleep(delay)

# tree txt
df = pd.read_csv(pathsf, dtype=str).fillna("").drop_duplicates("url").sort_values("path")
lines = [urlparse(root).path]
for p, t, s in zip(df["path"], df["type"], df["size"]):
    d = p.rstrip("/").count("/") - base_depth
    lines.append("    " * (d - 1) + p.rstrip("/").split("/")[-1] + ("/" if t == "dir" else f"  [{s}]" if s else ""))
treef.write_text("\n".join(lines), encoding="utf-8")

# summary
files = df[df["type"] == "file"]
print((df["type"] == "dir").sum(), "dirs,", len(files), "files,", round(pd.to_numeric(files["bytes"]).sum() / 1e9, 2), "GB approx from listing sizes")
print(files["path"].str.extract(r"(\.[A-Za-z0-9]+)$")[0].str.lower().value_counts().head(10).to_string())

6453 directories already done, 0 to go
6452 dirs, 294010 files, 23.48 GB approx from listing sizes
0
.pdf    148596
.tif    145394
.log        10
.zip         4
.txt         2


In [5]:
# setup
import csv, html, re, time, requests
import pandas as pd
from pathlib import Path
from urllib.parse import urljoin, unquote, urlparse

out = Path("data/labins/site/")
root = "https://ftp.labins.org/DRG2/"
name = "DRG2"
pathsf, logf, treef = out / f"{name}_paths.csv", out / f"{name}_log.csv", out / f"{name}_tree.txt"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}
delay = 1.5   # seconds between requests
cols = ["path", "url", "type", "size", "bytes", "modified", "depth"]
base_depth = urlparse(root).path.rstrip("/").count("/")

# patterns
link = re.compile(r'<a href="([^"]*)">(.*?)</a>(.*?)(?=<a href=|</pre>|</table>|\Z)', re.S | re.I)
stamp = re.compile(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}")
mult = {"K": 1024, "M": 1024**2, "G": 1024**3, "T": 1024**4}

# listing sizes like 55K to approx bytes
def tobytes(s):
    return float(s[:-1]) * mult[s[-1]] if s[-1:] in mult else float(s) if re.fullmatch(r"\d+(\.\d+)?", s) else 0

# parse one listing page
def parse(text, base):
    rows = []
    for href, label, tail in link.findall(text):
        if href.startswith("?") or "Parent Directory" in label:
            continue   # sort links and the way back up
        url = urljoin(base, html.unescape(href))
        if not url.startswith(base) or url == base:
            continue
        tail = html.unescape(re.sub(r"<[^>]+>", " ", tail)).replace("\xa0", " ")
        d = stamp.search(tail)
        rest = tail[d.end():].split() if d else []
        size = rest[0] if rest else ""
        path = unquote(urlparse(url).path)
        rows.append({"path": path, "url": url, "type": "dir" if url.endswith("/") else "file", "size": size,
                     "bytes": tobytes(size), "modified": d.group() if d else "", "depth": path.rstrip("/").count("/") - base_depth})
    return rows

# get with retry on 5xx
def get(url):
    for i in range(3):
        r = requests.get(url, headers=hdr, timeout=60)
        if r.status_code < 500:
            return r
        time.sleep(2 * (i + 1))
    return r

# load checkpoint
out.mkdir(parents=True, exist_ok=True)
if logf.exists() and pathsf.exists():
    log = pd.read_csv(logf, dtype=str).fillna("")
    done = set(log.loc[log["status"] == "200", "url"])
    known = pd.read_csv(pathsf, dtype=str).fillna("")
    seen = set(known.loc[known["type"] == "dir", "url"]) | {root}
else:
    pathsf.write_text(",".join(cols) + "\n")
    logf.write_text("url,status,dirs,files\n")
    done, seen = set(), {root}
stack = [u for u in sorted(seen, key=lambda u: (u != root, u)) if u not in done]
print(len(done), "directories already done,", len(stack), "to go")

# crawl
while stack:
    u = stack.pop(0)
    r = get(u)
    print(r.status_code, u)
    nd = nf = 0
    if r.status_code == 200:
        rows = parse(r.text, u)
        with open(pathsf, "a", newline="") as fh:   # checkpoint per directory
            csv.DictWriter(fh, cols).writerows(rows)
        for x in rows:
            nd, nf = nd + (x["type"] == "dir"), nf + (x["type"] == "file")
            if x["type"] == "dir" and x["url"] not in seen:
                seen.add(x["url"])
                stack.append(x["url"])
    with open(logf, "a", newline="") as fh:
        csv.writer(fh).writerow([u, r.status_code, nd, nf])
    if r.status_code >= 400:   # stop on any 4xx
        print("stopping at", u)
        break
    time.sleep(delay)

# tree txt
df = pd.read_csv(pathsf, dtype=str).fillna("").drop_duplicates("url").sort_values("path")
lines = [urlparse(root).path]
for p, t, s in zip(df["path"], df["type"], df["size"]):
    d = p.rstrip("/").count("/") - base_depth
    lines.append("    " * (d - 1) + p.rstrip("/").split("/")[-1] + ("/" if t == "dir" else f"  [{s}]" if s else ""))
treef.write_text("\n".join(lines), encoding="utf-8")

# summary
files = df[df["type"] == "file"]
print((df["type"] == "dir").sum(), "dirs,", len(files), "files,", round(pd.to_numeric(files["bytes"]).sum() / 1e9, 2), "GB approx from listing sizes")
print(files["path"].str.extract(r"(\.[A-Za-z0-9]+)$")[0].str.lower().value_counts().head(10).to_string())

39 directories already done, 0 to go
38 dirs, 3743 files, 16.16 GB approx from listing sizes
0
.exe    3742
.zip       1
